In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

2025-03-03 17:58:21.518942: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-03 17:58:22.528939: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-03 17:58:22.528992: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-03 17:58:22.777389: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-03 17:58:23.087396: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# load teh models required
frozen_model = tf.keras.models.load_model('models/pre_trained_CNN_architecture_CNN_LSTM_frozen.keras')
finetuned_model = tf.keras.models.load_model('models/pre_trained_CNN_architecture_CNN_LSTM_finetuned.keras')

2025-03-03 17:59:40.282284: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31134 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:86:00.0, compute capability: 7.0


In [3]:
# load a test image to get the height and width information
file_path = 'All_data/block_0103/all_np_files/Block0103_2020_08_26.npy'

loaded_test_image = np.load(file_path)

print(loaded_test_image.shape)
image_height = loaded_test_image.shape[0]
image_width = loaded_test_image.shape[1]
print(image_height, image_width)

(768, 1024, 3)
768 1024


In [4]:
def get_all_preds_in_test_time_series(block_name, main_data_folder, model):
    # get the path to the block
    path_to_block = os.path.join(main_data_folder, block_name)
    # load the numpy file with the feature sub-widows
    test_features = np.load(os.path.join(path_to_block, [file for file in os.listdir(path_to_block) if file[:9] == 'subwindow'][0]))
    # get the predicted values
    all_predicted_values = model.predict(test_features)

    # return the predicted values
    return(all_predicted_values)

In [5]:
def prediction_on_test_data(pred_values, image_height, image_width, stride = 8, kernel_size = 32):
    # density map
    Density_map = np.zeros((image_height, image_width))

    # counts map
    counts_map = np.zeros((image_height, image_width))
    
    # now, for every window, we will keep adding the values together and also add the counts
    counter = 0
#     need a counter to move into each predicted value in the pred values list
    for ii in range(0, image_height, stride):
        for jj in range(0, image_width, stride):
#         operations for density map
#             get the window of interest
            new_window = Density_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     fill each with the value c_k
            counts_window = np.full((new_window.shape[0], new_window.shape[1]), pred_values[counter])
#     get the shapes of this new window
            cw_height = counts_window.shape[0]
            cw_width = counts_window.shape[1]
#         Do c_k/r_2
            counts_window_new = counts_window/(cw_height*cw_width)
#     This is the value in the window now
            value_window = counts_window_new
#     place the values in the corrsponding area of the density map
            Density_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window + value_window

#         Let's now focus on capturing the counts of the windows
            new_window_c = counts_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     get the counts area
            count = np.ones((new_window_c.shape[0], new_window_c.shape[1]))
#     keep adding the counts to reflect the addition of densities
            counts_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window_c + count
#     increase the counter
            counter = counter + 1
            
#         get the normalized count
    normalized_counts = np.divide(Density_map, counts_map)
    
#     entire count on the test set
    pred_on_test = np.sum(normalized_counts)
    
#     return the predicted value
    return(pred_on_test, normalized_counts)


In [6]:
def get_final_forecasted_and_true_values(preds_from_model, im_height, im_weight, stride, kernel_size, csv_file_name, block_name):
    final_preds_list = []
    for i in range(7):
        preds_per_image = prediction_on_test_data(preds_from_model[:,i], im_height, im_weight, stride , kernel_size)
        final_preds_list.append(preds_per_image[0])
    # make this list  a dataframe
    preds_df = pd.DataFrame(final_preds_list, columns = ['Forecasted_value'])
    
    # Where do we have the true values?
    true_val_location = 'All_data/test_true_counts'
    true_value_file = pd.read_csv(os.path.join(true_val_location, csv_file_name))
    
    # compute the mae
    mae_value = mean_absolute_error(true_value_file[['True_count']], preds_df[['Forecasted_value']])
    # attach the true and the forecasted values together
    final_df = pd.concat((true_value_file, preds_df), axis = 1)
    # final df location
    final_loc = 'All_data/test_predicted_counts'
    # save this file
    final_df.to_csv(os.path.join(final_loc, block_name + '.csv'), index = False)

    return(final_preds_list, mae_value, final_df)

Block 0103

Predictions wth the frozen model

In [7]:
# first get the predictions
frozen_preds_block_0103 = get_all_preds_in_test_time_series('block_0103', 'All_data', frozen_model)

2025-03-03 17:59:59.013071: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


384/384 [==============================] - 10s 5ms/step


In [8]:
frozen_preds_block_0103.shape

(12288, 7)

In [9]:
frozen_final_forecasts_block_0103 = get_final_forecasted_and_true_values(frozen_preds_block_0103, image_height, image_width, 8, 32, 'true_counts_blk_0103.csv', 'block_0103')

In [10]:
frozen_normalized_forecasts_block_0103 = frozen_final_forecasts_block_0103[0]

In [11]:
print(frozen_normalized_forecasts_block_0103)

[32.15915139786005, 66.41855640358008, 83.4097794788868, 58.29196432335046, 43.084299861057225, 46.19095202510261, 21.408111328956814]


In [12]:
mae_frozen_block_0103 = frozen_final_forecasts_block_0103[1]
mae_frozen_block_0103

18.261184195022903

In [13]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0103[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0103_2020_08_26,40,40.000661,32.159151
1,Block0103_2020_08_27,39,39.000001,66.418556
2,Block0103_2020_08_28,41,41.000000,83.409779
3,Block0103_2020_08_31,31,31.000000,58.291964
4,Block0103_2020_09_02,32,32.000000,43.084300
5,Block0103_2020_09_07,40,40.002086,46.190952
6,Block0103_2020_09_16,27,27.000176,21.408111


Predictions with the finetuned model

In [14]:
# first get the predictions
finetuned_preds_block_0103 = get_all_preds_in_test_time_series('block_0103', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [15]:
finetuned_preds_block_0103.shape

(12288, 7)

In [16]:
finetuned_final_forecasts_block_0103 = get_final_forecasted_and_true_values(finetuned_preds_block_0103, image_height, image_width, 8, 32, 'true_counts_blk_0103.csv', 'block_0103')

In [17]:
finetuned_normalized_forecasts_block_0103 = finetuned_final_forecasts_block_0103[0]

In [18]:
print(finetuned_normalized_forecasts_block_0103)

[36.7731778057509, 80.37326844792203, 97.22520708504861, 69.70736792205811, 56.37871755481723, 55.45301987771403, 29.173701948747397]


In [19]:
mae_finetuned_block_0103 = finetuned_final_forecasts_block_0103[1]
mae_finetuned_block_0103

25.93401500436522

In [20]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0103[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0103_2020_08_26,40,40.000661,36.773178
1,Block0103_2020_08_27,39,39.000001,80.373268
2,Block0103_2020_08_28,41,41.000000,97.225207
3,Block0103_2020_08_31,31,31.000000,69.707368
4,Block0103_2020_09_02,32,32.000000,56.378718
5,Block0103_2020_09_07,40,40.002086,55.453020
6,Block0103_2020_09_16,27,27.000176,29.173702


Block 0104

Predictions wth the frozen model

In [21]:
# first get the predictions
frozen_preds_block_0104 = get_all_preds_in_test_time_series('block_0104', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [22]:
frozen_preds_block_0104.shape

(12288, 7)

In [23]:
frozen_final_forecasts_block_0104 = get_final_forecasted_and_true_values(frozen_preds_block_0104, image_height, image_width, 8, 32, 'true_counts_blk_0104.csv', 'block_0104')

In [24]:
frozen_normalized_forecasts_block_0104 = frozen_final_forecasts_block_0104[0]

In [25]:
print(frozen_normalized_forecasts_block_0104)

[19.315006810109555, 39.91338415568801, 53.15535446933022, 46.54907834196557, 28.189518834030434, 33.15941340942925, 14.911317565041248]


In [26]:
mae_frozen_block_0104 = frozen_final_forecasts_block_0104[1]
mae_frozen_block_0104

11.577508621196188

In [27]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0104[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0104_2020_08_26,33,33.000000,19.315007
1,Block0104_2020_08_27,30,30.000000,39.913384
2,Block0104_2020_08_28,39,39.000001,53.155354
3,Block0104_2020_08_31,40,40.000000,46.549078
4,Block0104_2020_09_02,41,40.998810,28.189519
5,Block0104_2020_09_07,42,42.169009,33.159413
6,Block0104_2020_09_16,30,30.005317,14.911318


Predictions with the finetuned model

In [28]:
# first get the predictions
finetuned_preds_block_0104 = get_all_preds_in_test_time_series('block_0104', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [29]:
finetuned_preds_block_0104.shape

(12288, 7)

In [30]:
finetuned_final_forecasts_block_0104 = get_final_forecasted_and_true_values(finetuned_preds_block_0104, image_height, image_width, 8, 32, 'true_counts_blk_0104.csv', 'block_0104')

In [31]:
finetuned_normalized_forecasts_block_0104 = finetuned_final_forecasts_block_0104[0]

In [32]:
print(finetuned_normalized_forecasts_block_0104)

[24.361620227453344, 50.954801853375514, 64.36772336218752, 55.05331849764886, 42.52060727405841, 41.56701725971248, 22.83665759703903]


In [33]:
mae_finetuned_block_0104 = finetuned_final_forecasts_block_0104[1]
mae_finetuned_block_0104

11.304450843295067

In [34]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0104[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0104_2020_08_26,33,33.000000,24.361620
1,Block0104_2020_08_27,30,30.000000,50.954802
2,Block0104_2020_08_28,39,39.000001,64.367723
3,Block0104_2020_08_31,40,40.000000,55.053318
4,Block0104_2020_09_02,41,40.998810,42.520607
5,Block0104_2020_09_07,42,42.169009,41.567017
6,Block0104_2020_09_16,30,30.005317,22.836658


Block 0105

Predictions wth the frozen model

In [35]:
# first get the predictions
frozen_preds_block_0105 = get_all_preds_in_test_time_series('block_0105', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [36]:
frozen_preds_block_0105.shape

(12288, 7)

In [37]:
frozen_final_forecasts_block_0105 = get_final_forecasted_and_true_values(frozen_preds_block_0105, image_height, image_width, 8, 32, 'true_counts_blk_0105.csv', 'block_0105')

In [38]:
frozen_normalized_forecasts_block_0105 = frozen_final_forecasts_block_0105[0]

In [39]:
print(frozen_normalized_forecasts_block_0105)

[26.312587172319432, 56.18159916881903, 71.07474441586665, 54.5519363294734, 36.14609239217772, 40.28016679148438, 17.778020234832283]


In [40]:
mae_frozen_block_0105 = frozen_final_forecasts_block_0105[1]
mae_frozen_block_0105

9.264535272330575

In [41]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0105[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0105_2020_08_26,40,40.000001,26.312587
1,Block0105_2020_08_27,46,46.002743,56.181599
2,Block0105_2020_08_28,58,58.000696,71.074744
3,Block0105_2020_08_31,41,41.000032,54.551936
4,Block0105_2020_09_02,41,41.001190,36.146092
5,Block0105_2020_09_07,36,36.000022,40.280167
6,Block0105_2020_09_16,23,23.000000,17.778020


Predictions with the finetuned model

In [42]:
# first get the predictions
finetuned_preds_block_0105 = get_all_preds_in_test_time_series('block_0105', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [43]:
finetuned_preds_block_0105.shape

(12288, 7)

In [44]:
finetuned_final_forecasts_block_0105 = get_final_forecasted_and_true_values(finetuned_preds_block_0105, image_height, image_width, 8, 32, 'true_counts_blk_0105.csv', 'block_0105')

In [45]:
finetuned_normalized_forecasts_block_0105 = finetuned_final_forecasts_block_0105[0]

In [46]:
print(finetuned_normalized_forecasts_block_0105)

[29.782999815898066, 68.01828105037234, 83.25197934317757, 62.97501921619794, 48.87091394478268, 48.394104400417625, 24.589137894546884]


In [47]:
mae_finetuned_block_0105 = finetuned_final_forecasts_block_0105[1]
mae_finetuned_block_0105

14.473776576228136

In [48]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0105[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0105_2020_08_26,40,40.000001,29.783000
1,Block0105_2020_08_27,46,46.002743,68.018281
2,Block0105_2020_08_28,58,58.000696,83.251979
3,Block0105_2020_08_31,41,41.000032,62.975019
4,Block0105_2020_09_02,41,41.001190,48.870914
5,Block0105_2020_09_07,36,36.000022,48.394104
6,Block0105_2020_09_16,23,23.000000,24.589138


Block 0106

Predictions wth the frozen model

In [49]:
# first get the predictions
frozen_preds_block_0106 = get_all_preds_in_test_time_series('block_0106', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [50]:
frozen_preds_block_0106.shape

(12288, 7)

In [51]:
frozen_final_forecasts_block_0106 = get_final_forecasted_and_true_values(frozen_preds_block_0106, image_height, image_width, 8, 32, 'true_counts_blk_0106.csv', 'block_0106')

In [52]:
frozen_normalized_forecasts_block_0106 = frozen_final_forecasts_block_0106[0]

In [53]:
print(frozen_normalized_forecasts_block_0106)

[24.46449442445737, 48.201139800818865, 60.22204535427562, 51.33768991991706, 33.28498964452592, 37.55709831291987, 15.559093289426373]


In [54]:
mae_frozen_block_0106 = frozen_final_forecasts_block_0106[1]
mae_frozen_block_0106

13.270742771954573

In [55]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0106[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0106_2020_08_26,39,38.999667,24.464494
1,Block0106_2020_08_27,39,38.999989,48.201140
2,Block0106_2020_08_28,45,45.000000,60.222045
3,Block0106_2020_08_31,40,39.986887,51.337690
4,Block0106_2020_09_02,43,42.999997,33.284990
5,Block0106_2020_09_07,48,47.996502,37.557098
6,Block0106_2020_09_16,38,38.000000,15.559093


Predictions with the finetuned model

In [56]:
# first get the predictions
finetuned_preds_block_0106 = get_all_preds_in_test_time_series('block_0106', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [57]:
finetuned_preds_block_0106.shape

(12288, 7)

In [58]:
finetuned_final_forecasts_block_0106 = get_final_forecasted_and_true_values(finetuned_preds_block_0106, image_height, image_width, 8, 32, 'true_counts_blk_0106.csv', 'block_0106')

In [59]:
finetuned_normalized_forecasts_block_0106 = finetuned_final_forecasts_block_0106[0]

In [60]:
print(finetuned_normalized_forecasts_block_0106)

[28.356891528741322, 59.44430377691252, 72.21323639770222, 60.42251197261145, 46.574536143111494, 45.85156758871178, 23.57712960218406]


In [61]:
mae_finetuned_block_0106 = finetuned_final_forecasts_block_0106[1]
mae_finetuned_block_0106

14.124142795814361

In [62]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0106[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0106_2020_08_26,39,38.999667,28.356892
1,Block0106_2020_08_27,39,38.999989,59.444304
2,Block0106_2020_08_28,45,45.000000,72.213236
3,Block0106_2020_08_31,40,39.986887,60.422512
4,Block0106_2020_09_02,43,42.999997,46.574536
5,Block0106_2020_09_07,48,47.996502,45.851568
6,Block0106_2020_09_16,38,38.000000,23.577130


Block 0201

Predictions wth the frozen model

In [63]:
# first get the predictions
frozen_preds_block_0201 = get_all_preds_in_test_time_series('block_0201', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [64]:
frozen_preds_block_0201.shape

(12288, 7)

In [65]:
frozen_final_forecasts_block_0201 = get_final_forecasted_and_true_values(frozen_preds_block_0201, image_height, image_width, 8, 32, 'true_counts_blk_0201.csv', 'block_0201')

In [66]:
frozen_normalized_forecasts_block_0201 = frozen_final_forecasts_block_0201[0]

In [67]:
print(frozen_normalized_forecasts_block_0201)

[31.043713741325515, 65.84327173424795, 81.60038213680134, 56.457758517290735, 42.38909297980331, 43.1995362908658, 20.88839293890305]


In [68]:
mae_frozen_block_0201 = frozen_final_forecasts_block_0201[1]
mae_frozen_block_0201

14.936847854111509

In [69]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0201[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0201_2020_08_26,45,45.000217,31.043714
1,Block0201_2020_08_27,45,45.000040,65.843272
2,Block0201_2020_08_28,47,47.000001,81.600382
3,Block0201_2020_08_31,38,38.000000,56.457759
4,Block0201_2020_09_02,42,42.000041,42.389093
5,Block0201_2020_09_07,35,35.000000,43.199536
6,Block0201_2020_09_16,29,29.000000,20.888393


Predictions with the finetuned model

In [70]:
# first get the predictions
finetuned_preds_block_0201 = get_all_preds_in_test_time_series('block_0201', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [71]:
finetuned_preds_block_0201.shape

(12288, 7)

In [72]:
finetuned_final_forecasts_block_0201 = get_final_forecasted_and_true_values(finetuned_preds_block_0201, image_height, image_width, 8, 32, 'true_counts_blk_0201.csv', 'block_0201')

In [73]:
finetuned_normalized_forecasts_block_0201 = finetuned_final_forecasts_block_0201[0]

In [74]:
print(finetuned_normalized_forecasts_block_0201)

[36.17218313398599, 79.41938927854223, 95.41815060014918, 66.9876693142869, 54.17054024317341, 52.314355532106795, 28.07077133446046]


In [75]:
mae_finetuned_block_0201 = finetuned_final_forecasts_block_0201[1]
mae_finetuned_block_0201

21.58102149997315

In [76]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0201[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0201_2020_08_26,45,45.000217,36.172183
1,Block0201_2020_08_27,45,45.000040,79.419389
2,Block0201_2020_08_28,47,47.000001,95.418151
3,Block0201_2020_08_31,38,38.000000,66.987669
4,Block0201_2020_09_02,42,42.000041,54.170540
5,Block0201_2020_09_07,35,35.000000,52.314356
6,Block0201_2020_09_16,29,29.000000,28.070771


Block 0202

Predictions wth the frozen model

In [77]:
# first get the predictions
frozen_preds_block_0202 = get_all_preds_in_test_time_series('block_0202', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [78]:
frozen_preds_block_0202.shape

(12288, 7)

In [79]:
frozen_final_forecasts_block_0202 = get_final_forecasted_and_true_values(frozen_preds_block_0202, image_height, image_width, 8, 32, 'true_counts_blk_0202.csv', 'block_0202')

In [80]:
frozen_normalized_forecasts_block_0202 = frozen_final_forecasts_block_0202[0]

In [81]:
print(frozen_normalized_forecasts_block_0202)

[30.767912095713353, 69.84756866497857, 85.62245117295517, 58.46012982055294, 49.51100886083043, 44.55571069273152, 19.434159879538733]


In [82]:
mae_frozen_block_0202 = frozen_final_forecasts_block_0202[1]
mae_frozen_block_0202

31.171277312471535

In [83]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0202[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0202_2020_08_26,18,17.999960,30.767912
1,Block0202_2020_08_27,21,21.000000,69.847569
2,Block0202_2020_08_28,23,23.000000,85.622451
3,Block0202_2020_08_31,21,20.999982,58.460130
4,Block0202_2020_09_02,21,21.000000,49.511009
5,Block0202_2020_09_07,18,18.000000,44.555711
6,Block0202_2020_09_16,18,18.000000,19.434160


Predictions with the finetuned model

In [84]:
# first get the predictions
finetuned_preds_block_0202 = get_all_preds_in_test_time_series('block_0202', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [85]:
finetuned_preds_block_0202.shape

(12288, 7)

In [86]:
finetuned_final_forecasts_block_0202 = get_final_forecasted_and_true_values(finetuned_preds_block_0202, image_height, image_width, 8, 32, 'true_counts_blk_0202.csv', 'block_0202')

In [87]:
finetuned_normalized_forecasts_block_0202 = finetuned_final_forecasts_block_0202[0]

In [88]:
print(finetuned_normalized_forecasts_block_0202)

[34.601731812593684, 82.17270852519508, 98.82052122151288, 69.68002707735486, 60.83035130328688, 54.3344303032904, 25.234718006962076]


In [89]:
mae_finetuned_block_0202 = finetuned_final_forecasts_block_0202[1]
mae_finetuned_block_0202

40.8106411785994

In [90]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0202[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0202_2020_08_26,18,17.999960,34.601732
1,Block0202_2020_08_27,21,21.000000,82.172709
2,Block0202_2020_08_28,23,23.000000,98.820521
3,Block0202_2020_08_31,21,20.999982,69.680027
4,Block0202_2020_09_02,21,21.000000,60.830351
5,Block0202_2020_09_07,18,18.000000,54.334430
6,Block0202_2020_09_16,18,18.000000,25.234718


Block 0205

Predictions wth the frozen model

In [91]:
# first get the predictions
frozen_preds_block_0205 = get_all_preds_in_test_time_series('block_0205', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [92]:
frozen_preds_block_0205.shape

(12288, 7)

In [93]:
frozen_final_forecasts_block_0205 = get_final_forecasted_and_true_values(frozen_preds_block_0205, image_height, image_width, 8, 32, 'true_counts_blk_0205.csv', 'block_0205')

In [94]:
frozen_normalized_forecasts_block_0205 = frozen_final_forecasts_block_0205[0]

In [95]:
print(frozen_normalized_forecasts_block_0205)

[26.226947941726767, 48.941133512792426, 61.15722397656271, 49.772972250213805, 30.933390066904604, 34.80190467513987, 16.57211720996993]


In [96]:
mae_frozen_block_0205 = frozen_final_forecasts_block_0205[1]
mae_frozen_block_0205

11.04813854940397

In [97]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0205[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0205_2020_08_26,44,44.000000,26.226948
1,Block0205_2020_08_27,42,42.000001,48.941134
2,Block0205_2020_08_28,45,45.000000,61.157224
3,Block0205_2020_08_31,43,43.000000,49.772972
4,Block0205_2020_09_02,39,39.000000,30.933390
5,Block0205_2020_09_07,41,40.999915,34.801905
6,Block0205_2020_09_16,32,31.999656,16.572117


Predictions with the finetuned model

In [98]:
# first get the predictions
finetuned_preds_block_0205 = get_all_preds_in_test_time_series('block_0205', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [99]:
finetuned_preds_block_0205.shape

(12288, 7)

In [100]:
finetuned_final_forecasts_block_0205 = get_final_forecasted_and_true_values(finetuned_preds_block_0205, image_height, image_width, 8, 32, 'true_counts_blk_0205.csv', 'block_0205')

In [101]:
finetuned_normalized_forecasts_block_0205 = finetuned_final_forecasts_block_0205[0]

In [102]:
print(finetuned_normalized_forecasts_block_0205)

[29.98484607835397, 58.95666067536472, 71.98751483278734, 58.90134293494822, 44.39646649306845, 43.86873533610722, 25.088189375443715]


In [103]:
mae_finetuned_block_0205 = finetuned_final_forecasts_block_0205[1]
mae_finetuned_block_0205

12.71966925978261

In [104]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0205[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0205_2020_08_26,44,44.000000,29.984846
1,Block0205_2020_08_27,42,42.000001,58.956661
2,Block0205_2020_08_28,45,45.000000,71.987515
3,Block0205_2020_08_31,43,43.000000,58.901343
4,Block0205_2020_09_02,39,39.000000,44.396466
5,Block0205_2020_09_07,41,40.999915,43.868735
6,Block0205_2020_09_16,32,31.999656,25.088189


Block 0206

Predictions wth the frozen model

In [105]:
# first get the predictions
frozen_preds_block_0206 = get_all_preds_in_test_time_series('block_0206', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [106]:
frozen_preds_block_0206.shape

(12288, 7)

In [107]:
frozen_final_forecasts_block_0206 = get_final_forecasted_and_true_values(frozen_preds_block_0206, image_height, image_width, 8, 32, 'true_counts_blk_0206.csv', 'block_0206')

In [108]:
frozen_normalized_forecasts_block_0206 = frozen_final_forecasts_block_0206[0]

In [109]:
print(frozen_normalized_forecasts_block_0206)

[30.590913581943216, 62.00143607840681, 76.51148851956816, 55.71164897484332, 37.655352956533996, 39.811048213212004, 18.864574069795307]


In [110]:
mae_frozen_block_0206 = frozen_final_forecasts_block_0206[1]
mae_frozen_block_0206

17.423519318630913

In [111]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0206[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0206_2020_08_26,41,40.999838,30.590914
1,Block0206_2020_08_27,42,41.998810,62.001436
2,Block0206_2020_08_28,39,39.000067,76.511489
3,Block0206_2020_08_31,32,32.000003,55.711649
4,Block0206_2020_09_02,25,25.000352,37.655353
5,Block0206_2020_09_07,23,23.000040,39.811048
6,Block0206_2020_09_16,18,18.000000,18.864574


Predictions with the finetuned model

In [112]:
# first get the predictions
finetuned_preds_block_0206 = get_all_preds_in_test_time_series('block_0206', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [113]:
finetuned_preds_block_0206.shape

(12288, 7)

In [114]:
finetuned_final_forecasts_block_0206 = get_final_forecasted_and_true_values(finetuned_preds_block_0206, image_height, image_width, 8, 32, 'true_counts_blk_0206.csv', 'block_0206')

In [115]:
finetuned_normalized_forecasts_block_0206 = finetuned_final_forecasts_block_0206[0]

In [116]:
print(finetuned_normalized_forecasts_block_0206)

[34.54919415199618, 74.0468568558554, 88.76702631557296, 65.07679712834377, 50.65890198934275, 48.9216466566403, 26.502755774360896]


In [117]:
mae_finetuned_block_0206 = finetuned_final_forecasts_block_0206[1]
mae_finetuned_block_0206

25.917827224017127

In [118]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0206[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0206_2020_08_26,41,40.999838,34.549194
1,Block0206_2020_08_27,42,41.998810,74.046857
2,Block0206_2020_08_28,39,39.000067,88.767026
3,Block0206_2020_08_31,32,32.000003,65.076797
4,Block0206_2020_09_02,25,25.000352,50.658902
5,Block0206_2020_09_07,23,23.000040,48.921647
6,Block0206_2020_09_16,18,18.000000,26.502756


Block 0302

Predictions wth the frozen model

In [119]:
# first get the predictions
frozen_preds_block_0302 = get_all_preds_in_test_time_series('block_0302', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [120]:
frozen_preds_block_0302.shape

(12288, 7)

In [121]:
frozen_final_forecasts_block_0302 = get_final_forecasted_and_true_values(frozen_preds_block_0302, image_height, image_width, 8, 32, 'true_counts_blk_0302.csv', 'block_0302')

In [122]:
frozen_normalized_forecasts_block_0302 = frozen_final_forecasts_block_0302[0]

In [123]:
print(frozen_normalized_forecasts_block_0302)

[31.674873176883764, 65.30880127925188, 80.81557960293584, 57.605367542915914, 41.80448596591018, 44.28038926085155, 21.646298809328865]


In [124]:
mae_frozen_block_0302 = frozen_final_forecasts_block_0302[1]
mae_frozen_block_0302

12.617671601732754

In [125]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0302[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0302_2020_08_26,49,49.000000,31.674873
1,Block0302_2020_08_27,49,49.000005,65.308801
2,Block0302_2020_08_28,54,53.999570,80.815580
3,Block0302_2020_08_31,50,50.042349,57.605368
4,Block0302_2020_09_02,43,43.000009,41.804486
5,Block0302_2020_09_07,48,48.000572,44.280389
6,Block0302_2020_09_16,37,36.999999,21.646299


Predictions with the finetuned model

In [126]:
# first get the predictions
finetuned_preds_block_0302 = get_all_preds_in_test_time_series('block_0302', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [127]:
finetuned_preds_block_0302.shape

(12288, 7)

In [128]:
finetuned_final_forecasts_block_0302 = get_final_forecasted_and_true_values(finetuned_preds_block_0302, image_height, image_width, 8, 32, 'true_counts_blk_0302.csv', 'block_0302')

In [129]:
finetuned_normalized_forecasts_block_0302 = finetuned_final_forecasts_block_0302[0]

In [130]:
print(finetuned_normalized_forecasts_block_0302)

[36.38505289286254, 78.14868055364794, 93.50017854388693, 68.58801564186598, 55.6435893992348, 53.70930237810777, 29.325722014006416]


In [131]:
mae_finetuned_block_0302 = finetuned_final_forecasts_block_0302[1]
mae_finetuned_block_0302

17.982713087124925

In [132]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0302[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0302_2020_08_26,49,49.000000,36.385053
1,Block0302_2020_08_27,49,49.000005,78.148681
2,Block0302_2020_08_28,54,53.999570,93.500179
3,Block0302_2020_08_31,50,50.042349,68.588016
4,Block0302_2020_09_02,43,43.000009,55.643589
5,Block0302_2020_09_07,48,48.000572,53.709302
6,Block0302_2020_09_16,37,36.999999,29.325722


Block 0303

Predictions wth the frozen model

In [133]:
# first get the predictions
frozen_preds_block_0303 = get_all_preds_in_test_time_series('block_0303', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [134]:
frozen_preds_block_0303.shape

(12288, 7)

In [135]:
frozen_final_forecasts_block_0303 = get_final_forecasted_and_true_values(frozen_preds_block_0303, image_height, image_width, 8, 32, 'true_counts_blk_0303.csv', 'block_0303')

In [136]:
frozen_normalized_forecasts_block_0303 = frozen_final_forecasts_block_0303[0]

In [137]:
print(frozen_normalized_forecasts_block_0303)

[27.959474930068456, 59.749683218120666, 75.43055356884724, 53.77713192171438, 38.90478275857345, 43.41720764032433, 20.41989813188792]


In [138]:
mae_frozen_block_0303 = frozen_final_forecasts_block_0303[1]
mae_frozen_block_0303

13.55714086366053

In [139]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0303[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0303_2020_08_26,49,49.000171,27.959475
1,Block0303_2020_08_27,46,46.000007,59.749683
2,Block0303_2020_08_28,43,42.999982,75.430554
3,Block0303_2020_08_31,39,38.999993,53.777132
4,Block0303_2020_09_02,36,36.025884,38.904783
5,Block0303_2020_09_07,36,36.000000,43.417208
6,Block0303_2020_09_16,23,23.000000,20.419898


Predictions with the finetuned model

In [140]:
# first get the predictions
finetuned_preds_block_0303 = get_all_preds_in_test_time_series('block_0303', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [141]:
finetuned_preds_block_0303.shape

(12288, 7)

In [142]:
finetuned_final_forecasts_block_0303 = get_final_forecasted_and_true_values(finetuned_preds_block_0303, image_height, image_width, 8, 32, 'true_counts_blk_0303.csv', 'block_0303')

In [143]:
finetuned_normalized_forecasts_block_0303 = finetuned_final_forecasts_block_0303[0]

In [144]:
print(finetuned_normalized_forecasts_block_0303)

[33.169044187646456, 73.6546651619357, 89.53595564009062, 65.2524111407838, 52.699922223239916, 52.24348906513114, 28.120252299463672]


In [145]:
mae_finetuned_block_0303 = finetuned_final_forecasts_block_0303[1]
mae_finetuned_block_0303

22.04823590614263

In [146]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0303[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0303_2020_08_26,49,49.000171,33.169044
1,Block0303_2020_08_27,46,46.000007,73.654665
2,Block0303_2020_08_28,43,42.999982,89.535956
3,Block0303_2020_08_31,39,38.999993,65.252411
4,Block0303_2020_09_02,36,36.025884,52.699922
5,Block0303_2020_09_07,36,36.000000,52.243489
6,Block0303_2020_09_16,23,23.000000,28.120252


Block 0305

Predictions wth the frozen model

In [147]:
# first get the predictions
frozen_preds_block_0305 = get_all_preds_in_test_time_series('block_0305', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [148]:
frozen_preds_block_0305.shape

(12288, 7)

In [149]:
frozen_final_forecasts_block_0305 = get_final_forecasted_and_true_values(frozen_preds_block_0305, image_height, image_width, 8, 32, 'true_counts_blk_0305.csv', 'block_0305')

In [150]:
frozen_normalized_forecasts_block_0305 = frozen_final_forecasts_block_0305[0]

In [151]:
print(frozen_normalized_forecasts_block_0305)

[32.14220827709971, 63.8592289039346, 78.90666475178021, 55.90389061968422, 40.19205908911611, 42.24848021541143, 18.855646014733]


In [152]:
mae_frozen_block_0305 = frozen_final_forecasts_block_0305[1]
mae_frozen_block_0305

18.58749561258484

In [153]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0305[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0305_2020_08_26,46,46.000000,32.142208
1,Block0305_2020_08_27,35,35.000000,63.859229
2,Block0305_2020_08_28,37,37.000000,78.906665
3,Block0305_2020_08_31,30,30.000652,55.903891
4,Block0305_2020_09_02,35,35.001190,40.192059
5,Block0305_2020_09_07,29,28.999994,42.248480
6,Block0305_2020_09_16,20,20.000018,18.855646


Predictions with the finetuned model

In [154]:
# first get the predictions
finetuned_preds_block_0305 = get_all_preds_in_test_time_series('block_0305', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [155]:
finetuned_preds_block_0305.shape

(12288, 7)

In [156]:
finetuned_final_forecasts_block_0305 = get_final_forecasted_and_true_values(finetuned_preds_block_0305, image_height, image_width, 8, 32, 'true_counts_blk_0305.csv', 'block_0305')

In [157]:
finetuned_normalized_forecasts_block_0305 = finetuned_final_forecasts_block_0305[0]

In [158]:
print(finetuned_normalized_forecasts_block_0305)

[37.184023303522345, 77.5599974240613, 93.02374904386085, 66.78109638481776, 53.45669199723891, 52.1609853699948, 26.857557762391988]


In [159]:
mae_finetuned_block_0305 = finetuned_final_forecasts_block_0305[1]
mae_finetuned_block_0305

27.52229352554904

In [160]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0305[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0305_2020_08_26,46,46.000000,37.184023
1,Block0305_2020_08_27,35,35.000000,77.559997
2,Block0305_2020_08_28,37,37.000000,93.023749
3,Block0305_2020_08_31,30,30.000652,66.781096
4,Block0305_2020_09_02,35,35.001190,53.456692
5,Block0305_2020_09_07,29,28.999994,52.160985
6,Block0305_2020_09_16,20,20.000018,26.857558


Block 0306

Predictions wth the frozen model

In [161]:
# first get the predictions
frozen_preds_block_0306 = get_all_preds_in_test_time_series('block_0306', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [162]:
frozen_preds_block_0306.shape

(12288, 7)

In [163]:
frozen_final_forecasts_block_0306 = get_final_forecasted_and_true_values(frozen_preds_block_0306, image_height, image_width, 8, 32, 'true_counts_blk_0306.csv', 'block_0306')

In [164]:
frozen_normalized_forecasts_block_0306 = frozen_final_forecasts_block_0306[0]

In [165]:
print(frozen_normalized_forecasts_block_0306)

[29.80074430337785, 56.27737978387208, 69.87221148007616, 54.25043049229294, 35.67850659022631, 39.06249301109772, 18.580144353014948]


In [166]:
mae_frozen_block_0306 = frozen_final_forecasts_block_0306[1]
mae_frozen_block_0306

11.223344032392815

In [167]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0306[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0306_2020_08_26,41,41.000009,29.800744
1,Block0306_2020_08_27,41,41.000003,56.277380
2,Block0306_2020_08_28,43,43.006851,69.872211
3,Block0306_2020_08_31,40,39.997604,54.250430
4,Block0306_2020_09_02,40,40.000000,35.678507
5,Block0306_2020_09_07,33,32.999982,39.062493
6,Block0306_2020_09_16,18,18.000000,18.580144


Predictions with the finetuned model

In [168]:
# first get the predictions
finetuned_preds_block_0306 = get_all_preds_in_test_time_series('block_0306', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [169]:
finetuned_preds_block_0306.shape

(12288, 7)

In [170]:
finetuned_final_forecasts_block_0306 = get_final_forecasted_and_true_values(finetuned_preds_block_0306, image_height, image_width, 8, 32, 'true_counts_blk_0306.csv', 'block_0306')

In [171]:
finetuned_normalized_forecasts_block_0306 = finetuned_final_forecasts_block_0306[0]

In [172]:
print(finetuned_normalized_forecasts_block_0306)

[34.498864311844734, 67.36698521750992, 81.12545726256798, 64.44649250388467, 49.31157437513845, 48.57794075700209, 27.03226747204584]


In [173]:
mae_finetuned_block_0306 = finetuned_final_forecasts_block_0306[1]
mae_finetuned_block_0306

18.480264753757748

In [174]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0306[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0306_2020_08_26,41,41.000009,34.498864
1,Block0306_2020_08_27,41,41.000003,67.366985
2,Block0306_2020_08_28,43,43.006851,81.125457
3,Block0306_2020_08_31,40,39.997604,64.446493
4,Block0306_2020_09_02,40,40.000000,49.311574
5,Block0306_2020_09_07,33,32.999982,48.577941
6,Block0306_2020_09_16,18,18.000000,27.032267
